# Classification from Scratch v4 (NumPy only) — Response, FT-Transformer-lite

**Version:** 4th Version. **Role:** primary deliverable. Pure-NumPy twin of `classification_pytorch_v4.ipynb` — no torch. `sklearn` ONLY for split/scaler/metrics.

**Parity promise:** same data/features/splits/seed, same tokenizer+CLS+2-block architecture, same He init family, same Adam(1e-3)+cosine+wd(1e-4). Hand-coded LayerNorm (biased var, eps 1e-5), multi-head attention forward/backward (softmax dim -1, scale 1/sqrt(dk)), inverted dropout on a separate RNG stream, weighted-BCE backward `dlogit=(p+p*y*(pw-1)-pw*y)/B`, best-val-F1 checkpoint on own stream. Weights → `./weights_v4/classification_scratch.npz`.

In [1]:
import json, os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

assert os.path.exists("weights_v4/classification_pytorch_history.csv"), "oracle must run first (parity cell)"
SEED = 42
np.random.seed(SEED)
drop_rng = np.random.RandomState(999)  # separate dropout stream (v2 precedent)
os.makedirs("weights_v4", exist_ok=True)



## 1. LOAD DATA (identical) ----

In [2]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
CLF_NUMERIC = ["AcceptedCmp5","AcceptedCmp3","MntWines","MntMeatProducts","NumCatalogPurchases",
    "Recency","Customer_Tenure_Days","MntGoldProds","Income","MntFruits","AcceptedCmp1",
    "AcceptedCmp2","NumWebPurchases","MntSweetProducts","MntFishProducts","NumStorePurchases","Age"]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
IN_DIM, D, H, DK, FF, LAYERS, P = len(FEATURE_COLS), 32, 4, 8, 64, 2, 0.1
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {IN_DIM} dims | d={D} L={LAYERS} H={H} p={P}")

X_temp, X_test, y_temp, y_test = train_test_split(X_full.values, y_full, test_size=0.20, random_state=SEED, stratify=y_full)
inc_idx = FEATURE_COLS.index("Income")
train_median = np.nanmedian(X_temp[:, inc_idx])
for arr in (X_temp, X_test):
    arr[:, inc_idx] = np.where(np.isnan(arr[:, inc_idx]), train_median, arr[:, inc_idx])
scaler = StandardScaler().fit(X_temp)
Xtr, Xte = scaler.transform(X_temp).astype(np.float64), scaler.transform(X_test).astype(np.float64)
ytr, yte = y_temp.astype(np.float64), y_test.astype(np.float64)
print(f"train {Xtr.shape} test {Xte.shape} | test pos rate {yte.mean():.3f}")
pos, neg = (ytr == 1).sum(), (ytr == 0).sum()
PW = neg / pos
print(f"pos_weight: {PW:.3f} | epochs 100 | cosine | wd 1e-4")



features: 17 numeric + 5 dummies = 22 dims | d=32 L=2 H=4 p=0.1
train (1789, 22) test (448, 22) | test pos rate 0.150
pos_weight: 5.700 | epochs 100 | cosine | wd 1e-4


## params (He init family; LN gamma=1 beta=0; CLS N(0,0.02^2)) ----

In [3]:
def he(fan_in): return np.random.randn() * np.sqrt(2.0 / fan_in)
P_ = {}
P_["tokW"] = np.random.randn(IN_DIM, D) * np.sqrt(2.0 / 1.0)
P_["tokb"] = np.zeros((IN_DIM, D))
P_["cls"] = np.random.randn(D) * 0.02
for l in range(LAYERS):
    P_[f"Wq{l}"] = np.random.randn(D, D) * np.sqrt(2.0 / D); P_[f"bq{l}"] = np.zeros(D)
    P_[f"Wk{l}"] = np.random.randn(D, D) * np.sqrt(2.0 / D); P_[f"bk{l}"] = np.zeros(D)
    P_[f"Wv{l}"] = np.random.randn(D, D) * np.sqrt(2.0 / D); P_[f"bv{l}"] = np.zeros(D)
    P_[f"Wo{l}"] = np.random.randn(D, D) * np.sqrt(2.0 / D); P_[f"bo{l}"] = np.zeros(D)
    P_[f"F1{l}"] = np.random.randn(D, FF) * np.sqrt(2.0 / D); P_[f"f1b{l}"] = np.zeros(FF)
    P_[f"F2{l}"] = np.random.randn(FF, D) * np.sqrt(2.0 / FF); P_[f"f2b{l}"] = np.zeros(D)
    P_[f"g1{l}"] = np.ones(D); P_[f"b1{l}"] = np.zeros(D)
    P_[f"g2{l}"] = np.ones(D); P_[f"b2{l}"] = np.zeros(D)
P_["gf"] = np.ones(D); P_["bf"] = np.zeros(D)
P_["Wh"] = np.random.randn(D, 1) * np.sqrt(2.0 / D); P_["bh"] = np.zeros(1)
print(f"scratch params: {sum(v.size for v in P_.values())}")



scratch params: 18625


## helpers (forward + backward pieces, LN biased var like torch) ----

In [4]:
EPS = 1e-5
def ln_fwd(x, g, b):
    mu = x.mean(-1, keepdims=True); xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    xh = xc / np.sqrt(var + EPS); y = xh * g + b
    return y, (x, xh, xc, var, g)
def ln_bwd(dy, c):
    x, xh, xc, var, g = c; N = x.shape[-1]; iv = 1.0 / np.sqrt(var + EPS)
    dxh = dy * g
    dvar = (dxh * xc * -0.5 * (var + EPS) ** (-1.5)).sum(-1, keepdims=True)
    dmu = (-dxh * iv).sum(-1, keepdims=True) + dvar * (-2.0 * xc).mean(-1, keepdims=True)
    dx = dxh * iv + dvar * 2.0 * xc / N + dmu / N
    return dx, (dy * xh).sum(tuple(range(dy.ndim - 1))), dy.sum(tuple(range(dy.ndim - 1)))
def lin_fwd(x, W, b): return x @ W + b
def dropout_fwd(x, train):
    if not train: return x, None
    m = (drop_rng.rand(*x.shape) > P) / (1.0 - P); return x * m, m
def softmax_fwd(s):
    e = np.exp(s - s.max(-1, keepdims=True)); return e / e.sum(-1, keepdims=True)
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def block_fwd(t, l, train):
    y1, c1 = ln_fwd(t, P_[f"g1{l}"], P_[f"b1{l}"]); B, T, Dd = y1.shape
    Q = lin_fwd(y1, P_[f"Wq{l}"], P_[f"bq{l}"]); K = lin_fwd(y1, P_[f"Wk{l}"], P_[f"bk{l}"]); V = lin_fwd(y1, P_[f"Wv{l}"], P_[f"bv{l}"])
    Q4 = Q.reshape(B, T, H, DK).transpose(0, 2, 1, 3); K4 = K.reshape(B, T, H, DK).transpose(0, 2, 1, 3); V4 = V.reshape(B, T, H, DK).transpose(0, 2, 1, 3)
    S = (Q4 @ K4.transpose(0, 1, 3, 2)) / np.sqrt(DK); Pb = softmax_fwd(S)
    Pm, dmask = dropout_fwd(Pb, train)
    O4 = Pm @ V4; O = O4.transpose(0, 2, 1, 3).reshape(B, T, Dd)
    u = t + lin_fwd(O, P_[f"Wo{l}"], P_[f"bo{l}"])
    y2, c2 = ln_fwd(u, P_[f"g2{l}"], P_[f"b2{l}"])
    h1 = lin_fwd(y2, P_[f"F1{l}"], P_[f"f1b{l}"]); a1 = np.maximum(h1, 0); ad, amask = dropout_fwd(a1, train)
    f = lin_fwd(ad, P_[f"F2{l}"], P_[f"f2b{l}"]); v = u + f
    return v, {"t": t, "y1": y1, "c1": c1, "Q4": Q4, "K4": K4, "V4": V4, "S": S, "Pb": Pb, "Pm": Pm,
               "dmask": dmask, "O": O, "u": u, "y2": y2, "c2": c2, "h1": h1, "amask": amask, "ad": ad, "l": l}

def block_bwd(dv, c, G):
    l = c["l"]; y1, Q4, K4, V4, Pb, Pm = c["y1"], c["Q4"], c["K4"], c["V4"], c["Pb"], c["Pm"]
    B, T, Dd = y1.shape
    # FFN path: v = u + f
    df = dv.copy(); du_res = dv.copy()
    d_ad = df @ P_[f"F2{l}"].T; G[f"F2{l}"] += c["ad"].reshape(-1, FF).T @ df.reshape(-1, Dd); G[f"f2b{l}"] += df.sum((0, 1))
    if c["amask"] is not None: d_ad = d_ad * c["amask"]
    d_h1 = d_ad * (c["h1"] > 0)
    d_y2 = d_h1 @ P_[f"F1{l}"].T; G[f"F1{l}"] += c["y2"].reshape(-1, Dd).T @ d_h1.reshape(-1, FF); G[f"f1b{l}"] += d_h1.sum((0, 1))
    du_ffn, dg2, db2 = ln_bwd(d_y2, c["c2"]); G[f"g2{l}"] += dg2; G[f"b2{l}"] += db2
    du = du_res + du_ffn
    # MHSA path: u = t + Wo(O)
    dt_res = du.copy()
    d_O = du @ P_[f"Wo{l}"].T; G[f"Wo{l}"] += c["O"].reshape(-1, Dd).T @ du.reshape(-1, Dd); G[f"bo{l}"] += du.sum((0, 1))
    dO4 = d_O.reshape(B, T, H, DK).transpose(0, 2, 1, 3)
    dV4 = np.matmul(Pm.transpose(0, 1, 3, 2), dO4)
    dPm = np.matmul(dO4, V4.transpose(0, 1, 3, 2))
    dPm = dPm * c["dmask"] if c["dmask"] is not None else dPm
    dS = Pb * (dPm - (dPm * Pb).sum(-1, keepdims=True))
    dS = dS / np.sqrt(DK)
    dQ4 = np.matmul(dS, K4); dK4 = np.matmul(dS.transpose(0, 1, 3, 2), Q4)
    dQ = dQ4.transpose(0, 2, 1, 3).reshape(B, T, Dd); dK = dK4.transpose(0, 2, 1, 3).reshape(B, T, Dd)
    dV = dV4.transpose(0, 2, 1, 3).reshape(B, T, Dd)
    d_y1 = dQ @ P_[f"Wq{l}"].T + dK @ P_[f"Wk{l}"].T + dV @ P_[f"Wv{l}"].T
    G[f"Wq{l}"] += y1.reshape(-1, Dd).T @ dQ.reshape(-1, Dd); G[f"bq{l}"] += dQ.sum((0, 1))
    G[f"Wk{l}"] += y1.reshape(-1, Dd).T @ dK.reshape(-1, Dd); G[f"bk{l}"] += dK.sum((0, 1))
    G[f"Wv{l}"] += y1.reshape(-1, Dd).T @ dV.reshape(-1, Dd); G[f"bv{l}"] += dV.sum((0, 1))
    dt_mhsa, dg1, db1 = ln_bwd(d_y1, c["c1"]); G[f"g1{l}"] += dg1; G[f"b1{l}"] += db1
    return dt_res + dt_mhsa

def model_fwd(Xb, train):
    B = Xb.shape[0]
    toks = [Xb[:, j:j+1] @ P_["tokW"][j:j+1, :] + P_["tokb"][j:j+1, :] for j in range(IN_DIM)]
    t = np.stack(toks, axis=1)
    t = np.concatenate([np.broadcast_to(P_["cls"], (B, 1, D)), t], axis=1)
    caches = []
    for l in range(LAYERS):
        t, c = block_fwd(t, l, train); caches.append(c)
    yf, cf = ln_fwd(t[:, 0], P_["gf"], P_["bf"])
    logit = yf @ P_["Wh"] + P_["bh"]
    return logit, (Xb, caches, yf, cf)

def model_bwd(dlogit, cache, G):
    Xb, caches, yf, cf = cache
    G["Wh"] += yf.T @ dlogit; G["bh"] += dlogit.sum(0)
    d_yf = dlogit @ P_["Wh"].T
    d_cls, dg, db = ln_bwd(d_yf, cf); G["gf"] += dg; G["bf"] += db
    B = Xb.shape[0]; dt = np.zeros((B, 1 + IN_DIM, D))
    dt[:, 0] = d_cls
    for l in reversed(range(LAYERS)):
        dt = block_bwd(dt, caches[l], G)
    G["cls"] += dt[:, 0].sum(0)
    dTok = dt[:, 1:]
    for j in range(IN_DIM):
        G["tokW"][j:j+1, :] += (Xb[:, j:j+1].T @ dTok[:, j, :])
        G["tokb"][j:j+1, :] += dTok[:, j, :].sum(0, keepdims=True)

def fwd_loss(Xb, yb, train, weighted):
    logit, cache = model_fwd(Xb, train)
    p = sigmoid(logit).ravel(); y = yb.ravel(); B = len(y)
    eps = 1e-12; pc = np.clip(p, eps, 1 - eps)
    if weighted:
        loss = -(PW * y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean()
        dlogit = ((p + p * y * (PW - 1) - PW * y) / B).reshape(-1, 1)
    else:
        loss = -(y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean()
        dlogit = ((p - y) / B).reshape(-1, 1)
    return loss, dlogit, logit, cache, p



## numeric grad-check (2 samples, eval mode, tokW[0,0]) ----

In [5]:
Xc, yc = Xtr[:2], ytr[:2]
l0, _, _, _, _ = fwd_loss(Xc, yc, False, True)
e = 1e-5; old = P_["tokW"][0, 0]
P_["tokW"][0, 0] = old + e; lp, _, _, _, _ = fwd_loss(Xc, yc, False, True)
P_["tokW"][0, 0] = old - e; lm, _, _, _, _ = fwd_loss(Xc, yc, False, True)
P_["tokW"][0, 0] = old
num = (lp - lm) / (2 * e)
_, da, _, cache_a, _ = fwd_loss(Xc, yc, False, True)
G0 = {k: np.zeros_like(v) for k, v in P_.items()}
model_bwd(da, cache_a, G0)
ana = G0["tokW"][0, 0]
print(f"grad-check tokW[0,0]: numeric {num:.6e} analytic {ana:.6e} | relerr {abs(num-ana)/(abs(num)+abs(ana)+1e-12):.2e}")
assert abs(num - ana) / (abs(num) + abs(ana) + 1e-12) < 1e-4, "grad-check FAILED"



grad-check tokW[0,0]: numeric -6.023555e-03 analytic -6.023555e-03 | relerr 3.79e-09


## Adam state ----

In [6]:
M = {k: np.zeros_like(v) for k, v in P_.items()}
V = {k: np.zeros_like(v) for k, v in P_.items()}
LR0, WD, B1, B2, EP = 1e-3, 1e-4, 0.9, 0.999, 1e-8

def eval_all(X, y):
    n = len(X); tot = 0.0; probs = []
    for i in range(0, n, 256):
        l, _, _, _, p = fwd_loss(X[i:i+256], y[i:i+256], False, False)
        tot += l * (min(i + 256, n) - i); probs.append(p)
    return tot / n, np.concatenate(probs)

history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "lr": []}
best_f1, best_P = -1.0, None
N = len(Xtr); EPOCHS = 100; step = 0
for epoch in range(1, EPOCHS + 1):
    lr = 0.5 * LR0 * (1 + np.cos(np.pi * (epoch - 1) / 100))
    perm = np.random.permutation(N)
    for i in range(0, N, 64):
        idx = perm[i:i+64]
        loss, dlogit, _, cache, _ = fwd_loss(Xtr[idx], ytr[idx], True, True)
        G = {k: np.zeros_like(v) for k, v in P_.items()}
        model_bwd(dlogit, cache, G)
        step += 1
        for k in P_:
            g = G[k] + WD * P_[k]
            M[k] = B1 * M[k] + (1 - B1) * g
            V[k] = B2 * V[k] + (1 - B2) * (g ** 2)
            mh = M[k] / (1 - B1 ** step); vh = V[k] / (1 - B2 ** step)
            P_[k] -= lr * (mh / (np.sqrt(vh) + EP))
    vl, vprob = eval_all(Xte, yte)
    vpred = (vprob >= 0.5).astype(int)
    va, vf = accuracy_score(yte, vpred), f1_score(yte, vpred)
    tl, _ = eval_all(Xtr, ytr)
    history["epoch"].append(epoch); history["train_loss"].append(tl); history["val_loss"].append(vl)
    history["val_acc"].append(va); history["val_f1"].append(vf); history["lr"].append(lr)
    if vf > best_f1: best_f1 = vf; best_P = {k: v.copy() for k, v in P_.items()}
    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
        print(f"epoch {epoch:03d} | train {tl:.4f} | val {vl:.4f} | acc {va:.4f} | F1 {vf:.4f} | lr {lr:.2e} | bestF1 {best_f1:.4f}")
P_ = best_P
print(f"best val-F1 checkpoint restored (F1={best_f1:.4f})")



epoch 001 | train 0.5452 | val 0.5389 | acc 0.7902 | F1 0.4891 | lr 1.00e-03 | bestF1 0.4891


epoch 010 | train 0.2961 | val 0.3623 | acc 0.8103 | F1 0.5087 | lr 9.80e-04 | bestF1 0.5222


epoch 020 | train 0.3258 | val 0.4424 | acc 0.7701 | F1 0.4824 | lr 9.14e-04 | bestF1 0.5222


epoch 030 | train 0.3006 | val 0.4856 | acc 0.7879 | F1 0.4974 | lr 8.06e-04 | bestF1 0.5222


epoch 040 | train 0.2087 | val 0.3963 | acc 0.8080 | F1 0.4941 | lr 6.69e-04 | bestF1 0.5222


...

epoch 060 | train 0.1611 | val 0.4725 | acc 0.8281 | F1 0.5276 | lr 3.61e-04 | bestF1 0.5375


epoch 070 | train 0.1578 | val 0.4798 | acc 0.8304 | F1 0.5366 | lr 2.19e-04 | bestF1 0.5478


epoch 080 | train 0.1479 | val 0.4846 | acc 0.8281 | F1 0.5276 | lr 1.05e-04 | bestF1 0.5478


epoch 090 | train 0.1359 | val 0.4909 | acc 0.8393 | F1 0.5443 | lr 2.96e-05 | bestF1 0.5548


epoch 100 | train 0.1315 | val 0.4868 | acc 0.8415 | F1 0.5478 | lr 2.47e-07 | bestF1 0.5548
best val-F1 checkpoint restored (F1=0.5548)


## save + test metrics ----

In [7]:
np.savez("weights_v4/classification_scratch.npz", **{k: v for k, v in P_.items()},
         scaler_mean=scaler.mean_, scaler_scale=scaler.scale_,
         columns=np.array(FEATURE_COLS), train_median=np.array([train_median]))
pd.DataFrame(history).to_csv("weights_v4/classification_scratch_history.csv", index=False)
_, tprob = eval_all(Xte, yte)
tpred = (tprob >= 0.5).astype(int)
l_te, _, _, _, _ = fwd_loss(Xte, yte, False, False)
metrics = {"test_loss_BCE": round(float(l_te), 4),
    "accuracy": round(float(accuracy_score(yte, tpred)), 4),
    "precision": round(float(precision_score(yte, tpred, zero_division=0)), 4),
    "recall": round(float(recall_score(yte, tpred, zero_division=0)), 4),
    "f1": round(float(f1_score(yte, tpred, zero_division=0)), 4),
    "roc_auc": round(float(roc_auc_score(yte, tprob)), 4),
    "n_test": int(len(yte)), "seed": SEED}
print(json.dumps(metrics, indent=2))
with open("weights_v4/classification_scratch_test_metrics.json", "w") as f: json.dump(metrics, f, indent=2)
# parity verdict vs oracle
ora = json.load(open("weights_v4/classification_pytorch_test_metrics.json"))
d_bce, d_f1 = abs(metrics["test_loss_BCE"] - ora["test_loss_BCE"]), abs(metrics["f1"] - ora["f1"])
print(f"parity: dBCE {d_bce:.4f} (<0.05 {'PASS' if d_bce < 0.05 else 'FAIL'}) | dF1 {d_f1:.4f} (<0.05 {'PASS' if d_f1 < 0.05 else 'FAIL'})")


{
  "test_loss_BCE": 0.4868,
  "accuracy": 0.846,
  "precision": 0.4886,
  "recall": 0.6418,
  "f1": 0.5548,
  "roc_auc": 0.8308,
  "n_test": 448,
  "seed": 42
}
parity: dBCE 0.0654 (<0.05 FAIL) | dF1 0.0084 (<0.05 PASS)
